In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from pathlib import Path

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

train_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(p=0.5),  # Data augmentation
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225])  # ImageNet normalization
])

val_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225])
])

data_dir = Path("C:\\Users\\mjaye\\PycharmProjects\\FairChess\\dataset")  # Update with your data path
full_dataset = datasets.ImageFolder(root=data_dir, transform=train_transforms)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

Using device: cuda


In [5]:
class ChessPieceCNN(nn.Module):
    def __init__(self, num_classes):
        super(ChessPieceCNN, self).__init__()
        
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(128 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, num_classes)
        
    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        
        # Flatten for FC layers
        x = x.view(-1, 128 * 8 * 8)
        
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        
        return x

class_names = full_dataset.classes
num_classes = len(class_names)
model = ChessPieceCNN(num_classes).to(device)

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs):
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
        
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()
        
        
        train_loss = running_loss / len(train_loader)
        train_acc = 100 * correct_train / total_train
        val_loss = val_loss / len(val_loader)
        val_acc = 100 * correct_val / total_val
        
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)
        
        scheduler.step()
        
        print(f'Epoch [{epoch+1}/{num_epochs}]')
        print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
        print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
        print('-' * 50)
    
    return train_losses, val_losses, train_accuracies, val_accuracies

In [7]:
num_epochs = 25
print("Starting training...")

train_losses, val_losses, train_accs, val_accs = train_model(
    model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs
)

Starting training...
Epoch [1/25]
Train Loss: 1.5512, Train Acc: 56.60%
Val Loss: 0.8893, Val Acc: 71.58%
--------------------------------------------------
Epoch [2/25]
Train Loss: 0.8144, Train Acc: 73.35%
Val Loss: 0.6221, Val Acc: 74.21%
--------------------------------------------------
Epoch [3/25]
Train Loss: 0.6234, Train Acc: 78.89%
Val Loss: 0.4584, Val Acc: 77.37%
--------------------------------------------------
Epoch [4/25]
Train Loss: 0.5074, Train Acc: 81.40%
Val Loss: 0.3403, Val Acc: 89.47%
--------------------------------------------------
Epoch [5/25]
Train Loss: 0.3956, Train Acc: 85.75%
Val Loss: 0.2675, Val Acc: 88.95%
--------------------------------------------------
Epoch [6/25]
Train Loss: 0.2816, Train Acc: 90.37%
Val Loss: 0.1850, Val Acc: 91.05%
--------------------------------------------------
Epoch [7/25]
Train Loss: 0.2920, Train Acc: 90.37%
Val Loss: 0.1730, Val Acc: 92.63%
--------------------------------------------------
Epoch [8/25]
Train Loss: 0.

In [8]:
def evaluate_model(model, test_loader, class_names):
    model.eval()
    correct = 0
    total = 0
    class_correct = [0] * len(class_names)
    class_total = [0] * len(class_names)
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            # Per-class accuracy
            c = (predicted == labels).squeeze()
            for i in range(labels.size(0)):
                label = labels[i]
                class_correct[label] += c[i].item()
                class_total[label] += 1
    
    # Overall accuracy
    overall_acc = 100 * correct / total
    print(f'Overall Test Accuracy: {overall_acc:.2f}%')
    
    # Per-class accuracy
    for i in range(len(class_names)):
        if class_total[i] > 0:
            acc = 100 * class_correct[i] / class_total[i]
            print(f'Accuracy of {class_names[i]}: {acc:.2f}%')

In [9]:
evaluate_model(model, val_loader, class_names)

Overall Test Accuracy: 97.89%
Accuracy of bb: 100.00%
Accuracy of bk: 50.00%
Accuracy of bn: 87.50%
Accuracy of bp: 100.00%
Accuracy of bq: 87.50%
Accuracy of br: 100.00%
Accuracy of e: 100.00%
Accuracy of wb: 100.00%
Accuracy of wk: 100.00%
Accuracy of wn: 100.00%
Accuracy of wp: 100.00%
Accuracy of wq: 100.00%
Accuracy of wr: 100.00%


In [10]:
torch.save(model.state_dict(), 'chess_piece_cnn.pth')
print("Model saved!")

Model saved!
